# Notebook 5: Physics-Informed Neural Networks

**UKACM Autumn School: AI for Computational Mechanics**

### The pivot

Notebooks 1 to 4 all had the same shape. Take 1485 microstructures that somebody had already
homogenised by finite element, show the network the inputs and the answers, and let it interpolate
between them. The physics entered only through whoever ran the simulations.

This notebook has no training data. Not a small amount, none. There is no table of inputs and
outputs, no train/test split, no labels. Instead the network is handed the governing differential
equation and told to satisfy it.

The problem is a simply supported Euler-Bernoulli beam:

$$EI\,\frac{d^4 w}{dx^4} = q(x), \qquad w(0)=w(L)=0, \qquad w''(0)=w''(L)=0$$

A neural network $w_\theta(x)$ takes a coordinate and returns a deflection. Automatic
differentiation gives exact derivatives of that network with respect to $x$, so the residual
$EI\,w_\theta'''' - q$ can be evaluated at any point we like. Driving that residual to zero, together
with the boundary terms, is the whole training signal.

### What you will do

1. Set the beam problem up and check the closed-form solution against a numerical solve.
2. Verify that repeated `torch.autograd.grad` really does give correct fourth derivatives.
3. Write the physics loss and the boundary loss, and see that their relative weight is a free
   parameter that you have to choose, badly or well.
4. Train the PINN, watch the residual being driven down along the beam, and take the trained
   network apart into the weighted activations it sums to make the solution.
5. Find out how few collocation points you can get away with, and what failure looks like.
6. Time a retrain for a different load. That number is the reason Notebook 6 exists.

### Contents

| Part | Topic |
|---|---|
| 1 | The idea, and the beam problem |
| 2 | Automatic differentiation to fourth order |
| 3 | The network, the residual, and the loss weighting (interactive) |
| 4 | Training, the residual field, and how the network builds the solution |
| 5 | Collocation points (interactive) |
| 6 | Where PINNs struggle |

### On data files

Notebooks 1 to 4 load the microstructure dataset. This one loads nothing. Everything below is
generated from the governing equation, so the notebook runs anywhere PyTorch is installed.

In [ ]:
# --- Setup -------------------------------------------------------------------
# Run this first.

import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.colors import LogNorm
from IPython.display import HTML, display
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, SelectionSlider

np.random.seed(0)
torch.manual_seed(0)

# Fourth-order autograd builds a deep, narrow graph. On a 2-core machine the
# threading overhead on tensors this small costs more than it saves, so pin to one thread.
torch.set_num_threads(1)

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10, "axes.grid": True,
    "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False,
    "animation.embed_limit": 60,
})

C_DATA, C_FIT, C_ALT, C_BAD = "#3B6EA5", "#C25E00", "#4C9A5E", "#A8323E"

print("torch", torch.__version__, "| numpy", np.__version__)
print("threads:", torch.get_num_threads())


---

# Part 1: The idea, and the beam problem

### The governing equation

For a uniform beam of constant cross-section the Euler-Bernoulli equation is

$$\boxed{\;EI\,\frac{d^4 w}{dx^4} \;=\; q(x)\;}$$

It is a fourth-order linear ordinary differential equation on $0 \le x \le L$. Every symbol in it:

| Symbol | Meaning | SI unit | Value used here |
|---|---|---|---|
| $x$ | coordinate along the beam axis | m | $0$ to $L$ |
| $w(x)$ | transverse deflection, positive in the direction of the load | m | unknown, solved for |
| $E$ | Young's modulus of the beam material | Pa, or N/m$^2$ | |
| $I$ | second moment of area of the cross-section | m$^4$ | |
| $EI$ | flexural rigidity, the two only ever appear together | N m$^2$ | $1$ |
| $q(x)$ | distributed transverse load per unit length | N/m | $q_0 = 1$, uniform |
| $L$ | span between the supports | m | $1$ |

The numerical values are all unity so the printed numbers stay readable. The shape of the solution
and the structure of the method are what carry over, not the magnitudes.

Three derived quantities follow from $w$ by differentiation, and all three are used later:

$$\theta(x) = \frac{dw}{dx}\ \ \text{[rad]}, \qquad
M(x) = -EI\,\frac{d^2 w}{dx^2}\ \ \text{[N m]}, \qquad
V(x) = -EI\,\frac{d^3 w}{dx^3}\ \ \text{[N]}$$

with $\theta$ the slope, $M$ the bending moment, sagging positive, and $V$ the shear force.

### The boundary conditions

A fourth-order equation needs four conditions. A simply supported beam gives two at each end:

$$w(0) = 0, \qquad w(L) = 0$$

$$\left.\frac{d^2 w}{dx^2}\right|_{x=0} = 0, \qquad
  \left.\frac{d^2 w}{dx^2}\right|_{x=L} = 0$$

What each one means physically.

$w(0)=0$ and $w(L)=0$ say the supports hold the beam at its original height. The beam cannot move
vertically where it rests on them. A pin and a roller both enforce this.

$w''(0)=0$ and $w''(L)=0$ say the bending moment vanishes at each end, because $M = -EI\,w''$.
Neither support resists rotation, so neither can transmit a moment into the beam. The beam is free
to rotate about each support, it simply cannot translate there. This is exactly what separates
simply supported from clamped, where the slope is held at zero instead of the curvature.

### The reference solution

For uniform load $q(x) = q_0$ this boundary value problem has the closed-form solution

$$\boxed{\;w_{\text{exact}}(x) \;=\; \frac{q_0}{24EI}\,x\left(L^3 - 2Lx^2 + x^3\right)\;}$$

with mid-span deflection $w(L/2) = 5q_0L^4/(384EI)$. **This is the reference solution**, and every
accuracy number below is an error against it, so it had better be right. The next cell solves the
same boundary value problem by finite differences and compares. If those two disagree, nothing
downstream means anything.

### How accuracy is reported

One number throughout: the relative $L_2$ error of a predicted deflection $\hat{w}$, evaluated on a
dense grid of $N_p$ points $x_j$ that the network was never trained on.

$$\boxed{\;e_{\text{rel}} \;=\;
\frac{\Bigl(\sum_{j=1}^{N_p}\bigl(\hat{w}(x_j) - w_{\text{exact}}(x_j)\bigr)^2\Bigr)^{1/2}}
     {\Bigl(\sum_{j=1}^{N_p} w_{\text{exact}}(x_j)^2\Bigr)^{1/2}}\;}$$

It is dimensionless. A value of $10^{-3}$ means the prediction is wrong by about 0.1 percent of the
size of the solution. Dividing by the norm of the exact solution is what makes it comparable across
load cases whose deflections differ in magnitude.

In [ ]:
# --- Problem definition ------------------------------------------------------
L, EI, q0 = 1.0, 1.0, 1.0

def q_uniform(x):
    return q0 * torch.ones_like(x)

def w_exact(x):
    '''Closed-form deflection, simply supported beam under uniform load.'''
    return (q0 / (24.0 * EI)) * x * (L**3 - 2.0*L*x**2 + x**3)

def M_exact(x):     # bending moment, M = -EI w''
    return (q0 / 2.0) * x * (L - x)

def V_exact(x):     # shear force, V = -EI w'''
    return q0 * (L/2.0 - x)

w_max_closed = 5*q0*L**4 / (384*EI)
print(f"mid-span deflection from the closed form: {w_exact(np.array([L/2]))[0]:.8f} m")
print(f"the standard 5qL^4/384EI:                 {w_max_closed:.8f} m")


In [ ]:
# --- Independent check: finite difference solve of the same BVP --------------
# EI w'''' = q with w = w'' = 0 at both ends splits into two Poisson problems:
#   u'' = q          with u(0) = u(L) = 0,   where u = EI w''
#   w'' = u / EI     with w(0) = w(L) = 0

n = 401
xg = np.linspace(0, L, n)
h  = xg[1] - xg[0]

A = (np.diag(-2*np.ones(n-2)) + np.diag(np.ones(n-3), 1) + np.diag(np.ones(n-3), -1)) / h**2

u_fd = np.zeros(n); u_fd[1:-1] = np.linalg.solve(A, q0*np.ones(n-2))
w_fd = np.zeros(n); w_fd[1:-1] = np.linalg.solve(A, u_fd[1:-1] / EI)

w_cf  = w_exact(xg)
rel_fd = np.linalg.norm(w_fd - w_cf) / np.linalg.norm(w_cf)
print(f"finite difference vs closed form, {n} nodes")
print(f"  max absolute difference : {np.max(np.abs(w_fd - w_cf)):.3e} m")
print(f"  relative L2 difference  : {rel_fd:.3e}")
print(f"  (second-order scheme, so this should fall off as h^2)")


The two agree to the discretisation error of the finite difference scheme. The closed form is
safe to use as ground truth.

### Collocation points

A PINN never assembles a mesh. It picks a set of points $\{x_i\}$ in the domain, the **collocation
points**, and asks that the residual of the governing equation be small at each of them. The points
carry no unknowns and no connectivity. They are just places where the equation is interrogated.

The schematic below shows the beam, the load, the supports and 25 collocation points. Part 5 asks
how many are actually needed.

In [ ]:
# --- Schematic: the beam, the load, and the collocation points ---------------
fig, (ax, ax2) = plt.subplots(2, 1, figsize=(10, 6.0),
                              gridspec_kw={"height_ratios": [1.45, 1]})

ax.set_xlim(-0.20, 1.18); ax.set_ylim(-0.50, 0.62); ax.axis("off"); ax.grid(False)

# the beam itself
ax.fill_between([0, L], -0.035, 0.035, color="#3a3a3a", zorder=3)
ax.text(0.5, 0.0, r"$EI$", ha="center", va="center", color="w", fontsize=10, zorder=7)

# distributed load q(x) = q0, arrows pointing down onto the beam
for xa in np.linspace(0.02, 0.98, 17):
    ax.annotate("", xy=(xa, 0.045), xytext=(xa, 0.33),
                arrowprops=dict(arrowstyle="-|>", color=C_DATA, lw=1.1))
ax.plot([0.02, 0.98], [0.33, 0.33], color=C_DATA, lw=2)
ax.text(0.5, 0.44, r"$q(x) = q_0$   [N/m]", ha="center", color=C_DATA, fontsize=11)

# supports: pin and roller, both simply supported
for xp in [0.0, L]:
    ax.add_patch(plt.Polygon([[xp, -0.035], [xp-0.045, -0.135], [xp+0.045, -0.135]],
                             color="#555", zorder=5))
    ax.plot([xp-0.080, xp+0.080], [-0.160, -0.160], "k-", lw=2)
    for xh in np.linspace(xp-0.070, xp+0.070, 5):
        ax.plot([xh, xh-0.030], [-0.160, -0.215], "k-", lw=0.8)
ax.text(0.0, -0.285, "$w=0$\n$w''=0$", ha="center", va="top", fontsize=9)
ax.text(L,   -0.285, "$w=0$\n$w''=0$", ha="center", va="top", fontsize=9)

# span dimension L
ax.annotate("", xy=(0.0, -0.435), xytext=(L, -0.435),
            arrowprops=dict(arrowstyle="<|-|>", color="k", lw=1.1))
ax.plot([0.0, 0.0], [-0.160, -0.435], "k:", lw=0.7)
ax.plot([L, L],     [-0.160, -0.435], "k:", lw=0.7)
ax.text(0.5, -0.415, "$L$", ha="center", va="bottom", fontsize=11,
        bbox=dict(fc="w", ec="none", pad=1.0))

# coordinate axes, origin at the left support
ax.annotate("", xy=(0.26, 0.215), xytext=(0.0, 0.215),
            arrowprops=dict(arrowstyle="-|>", color=C_BAD, lw=1.4))
ax.text(0.275, 0.215, "$x$", color=C_BAD, fontsize=11, va="center")
ax.annotate("", xy=(0.0, 0.070), xytext=(0.0, 0.215),
            arrowprops=dict(arrowstyle="-|>", color=C_BAD, lw=1.4))
ax.text(-0.050, 0.125, "$w$", color=C_BAD, fontsize=11, va="center", ha="right")
ax.plot([0.0], [0.215], "o", ms=4, color=C_BAD)
ax.text(-0.050, 0.215, "$x=0$", color=C_BAD, fontsize=8.5, ha="right", va="center")

# collocation points marked along the beam axis
x_demo = np.linspace(0, L, 25)
ax.scatter(x_demo[1:-1], np.full(len(x_demo)-2, -0.075), s=22, color=C_FIT,
           zorder=8, marker="|", linewidths=1.6)
ax.scatter(x_demo[1:-1], np.full(len(x_demo)-2, -0.075), s=16, color=C_FIT, zorder=8)
ax.text(0.5, -0.135, "collocation points $x_i$", ha="center", color=C_FIT, fontsize=9)
ax.set_title("Simply supported Euler-Bernoulli beam under uniform load", fontsize=11)

# lower panel: the same points against the solution they are used to find
ax2.plot(xg, w_cf, color=C_ALT, lw=2, label="exact deflection $w_{exact}(x)$")
ax2.scatter(x_demo, np.zeros_like(x_demo), s=28, color=C_FIT, zorder=5,
            label="25 collocation points $x_i$")
ax2.scatter([0, L], [0, 0], s=90, marker="s", color=C_BAD, zorder=6,
            label="boundary points $x=0,\\,L$")
ax2.invert_yaxis()
ax2.set_xlim(-0.20, 1.18)
ax2.set_xlabel("$x$  (m)"); ax2.set_ylabel("$w$  (m)")
ax2.set_title("the residual is evaluated at the orange points, nowhere else", fontsize=10)
ax2.legend(fontsize=8, loc="lower left", framealpha=0.95)
plt.tight_layout(); plt.show()


**What the schematic shows.** Top: the problem this notebook solves. The beam spans $L$, the
load $q_0$ acts downwards along its whole length, and the two triangular supports are simply
supported, which is the $w=0,\ w''=0$ pair written beside each of them. The $x$ axis runs from the
left support and $w$ is measured downwards, positive with the load. The orange ticks on the beam
axis are the collocation points $x_i$, the only places where the differential equation will be
checked.

Bottom: the same 25 points drawn against the solution they are used to find. They sit on the axis,
not on the curve, because the PINN is never told what $w$ is at those points. It is only told what
the equation must do there. That is the entire difference between this notebook and Notebooks 1 to
4, where every training point came with an answer attached.

---

# Part 2: Automatic differentiation to fourth order

The whole method rests on one claim: `torch.autograd.grad`, applied repeatedly with
`create_graph=True`, returns exact derivatives of the network output with respect to its input.
Exact, not finite differenced. No step size, no truncation error.

That claim is easy to test. Take a function whose derivatives are known on paper, build it out of
torch operations, and differentiate it four times.

$$f(x) = \sin(3x) \quad\Rightarrow\quad f' = 3\cos 3x,\;\; f'' = -9\sin 3x,\;\; f''' = -27\cos 3x,
\;\; f'''' = 81\sin 3x$$

`create_graph=True` is the part that matters. It tells autograd to keep the derivative itself
differentiable, which is what lets the chain be repeated.

In [ ]:
# --- One derivative, then four ------------------------------------------------
def d_dx(y, x):
    '''dy/dx, keeping the result differentiable so it can be differentiated again.'''
    return torch.autograd.grad(y, x, grad_outputs=torch.ones_like(y), create_graph=True)[0]

k = 3.0
xt = torch.linspace(0, 2.0, 200).reshape(-1, 1).requires_grad_(True)

f  = torch.sin(k * xt)
d1 = d_dx(f,  xt)
d2 = d_dx(d1, xt)
d3 = d_dx(d2, xt)
d4 = d_dx(d3, xt)

xn = xt.detach().numpy().ravel()
analytic = [np.sin(k*xn), k*np.cos(k*xn), -k**2*np.sin(k*xn),
            -k**3*np.cos(k*xn), k**4*np.sin(k*xn)]
computed = [v.detach().numpy().ravel() for v in [f, d1, d2, d3, d4]]

print("order   max |autograd - analytical|")
for i, (a, c) in enumerate(zip(analytic, computed)):
    print(f"  {i}     {np.max(np.abs(a - c)):.3e}")


In [ ]:
# --- The same thing as a picture ---------------------------------------------
fig, axes = plt.subplots(1, 5, figsize=(15, 2.9), sharex=True)
names = ["$f$", "$f'$", "$f''$", "$f'''$", "$f''''$"]
for ax, a, c, nm in zip(axes, analytic, computed, names):
    ax.plot(xn, a, color=C_ALT, lw=3, alpha=0.55, label="analytical")
    ax.plot(xn, c, color=C_BAD, lw=1.2, ls="--", label="autograd")
    ax.set_title(nm, fontsize=11); ax.set_xlabel("$x$")
axes[0].legend(fontsize=8)
plt.suptitle("Repeated torch.autograd.grad on $\\sin(3x)$", y=1.06)
plt.tight_layout(); plt.show()


**What the five panels show.** Each panel is one order of differentiation, from the function
on the left to its fourth derivative on the right. The thick green line is the analytical
derivative, written on paper; the thin dashed red line is what repeated `torch.autograd.grad`
returned. They lie on top of each other in all five panels, and the table above gives the difference
as a number. Autograd is not approximating the derivative, it is applying the chain rule to the
exact expression graph.

Agreement to single-precision round-off at every order, against derivatives whose magnitude reaches
$3^4 = 81$. The cost is not free: each call adds another layer to the computation graph, so the
fourth derivative of a network costs well over an order of magnitude more than a plain forward pass.
That cost is measured in Part 6.

One consequence for the architecture. The activation has to be smooth enough to survive four
differentiations. ReLU is piecewise linear, so its second derivative is zero almost everywhere and
the residual would be identically zero for the wrong reason. `tanh` is $C^\infty$ and is the
standard choice for PINNs.

---

# Part 3: The network, the residual, and the weighting problem

### The network is the solution

In Notebooks 1 to 4 the network mapped a sample to a label. It took a microstructure, or a
descriptor vector, and returned a property. Here the network **is** the unknown field. It maps a
position along the beam to the deflection at that position:

$$\boxed{\;w_\theta : x \mapsto w_\theta(x), \qquad x \in [0, L],\ \ w_\theta(x) \ \text{in m}\;}$$

$\theta$ collects every weight and bias. The input $x$ is a **coordinate, not a data sample**. There
is no dataset with $x$ as a feature and $w$ as a label, because we do not know $w$ anywhere. That
single change is what separates a PINN from everything in Notebooks 1 to 4: the network plays the
role a finite element basis expansion plays in a classical solver, and $\theta$ plays the role of
the nodal unknowns.

Concretely, with $L_h$ hidden layers of width $H$ and $\tanh$ activation,

$$h^{(0)} = x, \qquad h^{(l)} = \tanh\!\left(W^{(l)} h^{(l-1)} + b^{(l)}\right), \quad l = 1 \ldots L_h,
\qquad w_\theta(x) = W^{(L_h+1)} h^{(L_h)} + b^{(L_h+1)}$$

$\tanh$ rather than ReLU because the loss below needs the fourth derivative, and a piecewise linear
activation has zero second derivative almost everywhere.

### The residual

Substitute $w_\theta$ into the governing equation and move everything to one side. What is left over
is the residual, a function of position with units of N/m:

$$\boxed{\;r(x) \;=\; EI\,\frac{d^4 w_\theta}{dx^4}(x) \;-\; q(x)\;}$$

If $r(x) = 0$ everywhere then $w_\theta$ satisfies the differential equation exactly. The fourth
derivative is the one Part 2 just verified autograd can compute, so $r$ is computable at any $x$ we
choose, for any $\theta$, with no mesh and no finite differences.

### Two losses

The **physics loss** is the mean squared residual over the $N_c$ collocation points:

$$L_{\text{phys}}(\theta) = \frac{1}{N_c}\sum_{i=1}^{N_c} r(x_i)^2
= \frac{1}{N_c}\sum_{i=1}^{N_c}
\left(EI\,w_\theta''''(x_i) - q(x_i)\right)^2$$

The **boundary loss** collects the four end conditions from Part 1, two deflections and two
curvatures:

$$L_{\text{bc}}(\theta) = \tfrac{1}{2}\left[w_\theta(0)^2 + w_\theta(L)^2\right]
+ \tfrac{1}{2}\left[w_\theta''(0)^2 + w_\theta''(L)^2\right]$$

and the two are combined into the single scalar that is actually minimised, with a weighting factor
$\lambda$ on the boundary term:

$$\boxed{\;L(\theta) \;=\; L_{\text{phys}}(\theta)
\;+\; \lambda\,L_{\text{bc}}(\theta),
\qquad \theta^\star = \arg\min_\theta L(\theta)\;}$$

That is the whole training signal. No labels appear anywhere in it.

### Why $\lambda$ is a problem

Look at the units. $L_{\text{phys}}$ is a squared load per unit length, of order $q_0^2$.
The deflection part of $L_{\text{bc}}$ is a squared length, of order $(q_0L^4/EI)^2$. The
two terms are not commensurable, so $\lambda$ is not dimensionless and there is no principled way to
add the terms without choosing it.

So $\lambda$ has to be picked by hand. Nothing in the method picks it for you. This is the honest
weak point of PINNs: an extra hyperparameter that decides whether the answer is right, sitting where
a classical solver would have had no choice to make at all. Adaptive weighting schemes exist and are
an active research area, but they are heuristics, not a fix.

The slider below makes the consequences concrete.

In [ ]:
# --- Network and losses -------------------------------------------------------
class BeamNet(nn.Module):
    '''Fully connected net, x -> w(x). tanh throughout so four derivatives survive.'''
    def __init__(self, hidden=32, n_layers=3, seed=0):
        super().__init__()
        torch.manual_seed(seed)
        layers = [nn.Linear(1, hidden), nn.Tanh()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden, hidden), nn.Tanh()]
        layers += [nn.Linear(hidden, 1)]
        self.net = nn.Sequential(*layers)
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x):
        return self.net(x)

def derivatives(model, x, order=4):
    '''Return [w, w', w'', ...] up to the requested order.'''
    out = [model(x)]
    for _ in range(order):
        out.append(d_dx(out[-1], x))
    return out

def physics_loss(model, x_col, q_fn):
    d = derivatives(model, x_col, order=4)
    return torch.mean((EI * d[4] - q_fn(x_col)) ** 2)

def bc_loss(model, x_bc):
    d = derivatives(model, x_bc, order=2)
    return torch.mean(d[0] ** 2) + torch.mean(d[2] ** 2)

_net = BeamNet()
n_par = sum(p.numel() for p in _net.parameters())
print("the network w_theta(x), layer by layer:")
for nm, m in _net.net.named_children():
    if isinstance(m, nn.Linear):
        print(f"   Linear({m.in_features} -> {m.out_features})   "
              f"{m.weight.numel()} weights + {m.bias.numel()} biases")
    else:
        print(f"   tanh")
print(f"\n   x (1 number, a coordinate)  ->  w_theta(x) (1 number, a deflection in m)")
print(f"   total trainable parameters theta : {n_par}")
print()
print("For comparison, the CNN in Notebook 3 had tens of thousands.")
print("PINNs are small. The cost is in the derivatives, not the width.")


In [ ]:
# --- The two losses have completely different magnitudes ---------------------
x_col_demo = torch.linspace(0, L, 100).reshape(-1, 1).requires_grad_(True)
x_bc_demo  = torch.tensor([[0.0], [L]]).requires_grad_(True)

lp0 = physics_loss(_net, x_col_demo, q_uniform).item()
lb0 = bc_loss(_net, x_bc_demo).item()
print(f"at initialisation:   L_phys = {lp0:.3e}      L_bc = {lb0:.3e}")
print(f"ratio L_phys / L_bc = {lp0/lb0:.3e}")
print()
print("The correct answer has deflections of order 1e-2 m, so L_bc for a")
print("badly converged net is small in absolute terms even when the boundary")
print("conditions are meaningfully violated. Weighting it too low is easy to do")
print("by accident.")


### Precompute solutions at six weights

Each of the six runs below is 1500 Adam iterations on the same network initialisation, the same 100
collocation points, and the same everything except $\lambda$. The cell prints its own timing.

Fixed budget is the point. A PINN in practice is trained for as long as you can afford, and the
question is what you get for that budget, not what you would get in the limit.

In [ ]:
# --- Train once per weight, store the results --------------------------------
LAMBDAS = [1e-2, 1e-1, 1e0, 1e1, 1e3, 1e5]
SWEEP_ITERS = 1500

x_plot   = np.linspace(0, L, 300)
w_plot_x = torch.from_numpy(x_plot).float().reshape(-1, 1)
w_plot_e = w_exact(x_plot)

def train_adam(model, x_col, x_bc, lam, iters, q_fn=q_uniform, lr=2e-3,
               record=None, record_res=None):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = {"total": [], "phys": [], "bc": []}
    for it in range(iters):
        opt.zero_grad()
        lp = physics_loss(model, x_col, q_fn)
        lb = bc_loss(model, x_bc)
        lt = lp + lam * lb
        lt.backward(); opt.step()
        hist["total"].append(lt.item()); hist["phys"].append(lp.item()); hist["bc"].append(lb.item())
        if record is not None and it % record[0] == 0:
            with torch.no_grad():
                record[1].append((it, model(w_plot_x).numpy().ravel()))
            if record_res is not None:
                dd = derivatives(model, x_col, order=4)
                record_res.append((EI * dd[4] - q_fn(x_col)).detach().numpy().ravel())
    return hist

def rel_l2(model):
    with torch.no_grad():
        wp = model(w_plot_x).numpy().ravel()
    return np.linalg.norm(wp - w_plot_e) / np.linalg.norm(w_plot_e), wp

x_col_s = torch.linspace(0, L, 100).reshape(-1, 1).requires_grad_(True)
x_bc_s  = torch.tensor([[0.0], [L]]).requires_grad_(True)

sweep = {}
t_sweep = time.time()
for lam in LAMBDAS:
    m = BeamNet(seed=0)
    h = train_adam(m, x_col_s, x_bc_s, lam, SWEEP_ITERS)
    r, wp = rel_l2(m)
    sweep[lam] = {"w": wp, "rel": r, "hist": h,
                  "phys": h["phys"][-1], "bc": h["bc"][-1]}
    print(f"  lambda = {lam:8.0e}   relative L2 error = {r:.3e}   "
          f"L_phys = {h['phys'][-1]:.2e}   L_bc = {h['bc'][-1]:.2e}", flush=True)
print(f"\n{len(LAMBDAS)} runs x {SWEEP_ITERS} iterations in {time.time()-t_sweep:.1f} s")


Move the slider. The left panel is the deflection, the right is the training history split into
its two parts.

At $\lambda = 10^{-2}$ the boundary conditions are barely penalised, so the network satisfies the
differential equation with the wrong constants of integration. The shape is a quartic, the residual
is small, and the answer is useless. That is the case worth staring at: **a small physics loss is
not evidence of a correct solution**.

At $\lambda = 10^5$ the boundary terms dominate the gradient and the interior equation is
effectively ignored within the budget.

In [ ]:
# --- Interactive: the loss weight ---------------------------------------------
def show_weight(log10_lambda=0):
    lam = 10.0 ** log10_lambda
    d = sweep[lam]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 4))

    a1.plot(x_plot, w_plot_e, color=C_ALT, lw=3, alpha=0.6, label="analytical")
    a1.plot(x_plot, d["w"],   color=C_BAD, lw=2, ls="--", label="PINN")
    a1.scatter([0, L], [0, 0], s=70, marker="s", color="k", zorder=5, label="required $w=0$")
    a1.invert_yaxis()
    a1.set_xlabel("$x$  (m)"); a1.set_ylabel("$w$  (m)")
    a1.set_title(f"$\\lambda$ = {lam:.0e}   relative $L_2$ error = {d['rel']:.2e}", fontsize=10)
    a1.legend(fontsize=8, loc="lower center")

    it = np.arange(1, SWEEP_ITERS + 1)
    a2.semilogy(it, d["hist"]["phys"], color=C_FIT, lw=1.4, label="$\L_{phys}$")
    a2.semilogy(it, d["hist"]["bc"],   color=C_DATA, lw=1.4, label="$\L_{bc}$")
    a2.set_xlabel("Adam iteration"); a2.set_ylabel("loss component")
    a2.set_ylim(1e-12, 1e2)
    a2.set_title("the two terms, unweighted", fontsize=10); a2.legend(fontsize=8)
    plt.tight_layout(); plt.show()

interact(show_weight,
         log10_lambda=IntSlider(0, min=-2, max=5, step=1, continuous_update=False,
                                description="log10 lambda"));


In [ ]:
# --- The U shape, all six weights at once ------------------------------------
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 4))
rels = [sweep[l]["rel"] for l in LAMBDAS]
a1.loglog(LAMBDAS, rels, "o-", color=C_FIT, lw=2, ms=8)
best = LAMBDAS[int(np.argmin(rels))]
a1.axvline(best, color=C_ALT, ls="--", lw=1.5, label=f"best of those tried: {best:.0e}")
a1.set_xlabel("$\\lambda$"); a1.set_ylabel("relative $L_2$ error")
a1.set_title(f"error spans {max(rels)/min(rels):.0f}x across the weights tried", fontsize=10)
a1.legend(fontsize=8)

for lam, col in zip(LAMBDAS, plt.cm.viridis(np.linspace(0, 0.9, len(LAMBDAS)))):
    a2.plot(x_plot, sweep[lam]["w"], lw=1.6, color=col, label=f"{lam:.0e}")
a2.plot(x_plot, w_plot_e, "k-", lw=2.5, label="exact")
a2.invert_yaxis(); a2.set_ylim(0.06, -0.06)
a2.set_xlabel("$x$  (m)"); a2.set_ylabel("$w$  (m)")
a2.set_title("same network, same points, same budget", fontsize=10)
a2.legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.show()


**What the two panels show.** Left: the relative $L_2$ error against $\lambda$ on log axes,
with the best of the six marked. The curve is U-shaped, and the spread printed in the panel title is
how much the answer changes when only this one number changes. Right: all six deflections drawn
against the exact solution, same network, same collocation points, same iteration budget.

The left-hand end of the U is the one to worry about. Those runs have a small physics loss and a
badly wrong answer, because the network is solving the differential equation with the wrong
constants of integration. Any quartic whose fourth derivative equals $q/EI$ satisfies the interior
equation; only one of them also sits on the supports. The right-hand end fails the other way,
spending its whole budget on the four boundary values and never fitting the interior.

No reading of the training loss would have told you which of these six runs to trust. Only the
comparison against the reference solution does, and on a real problem you do not have one.

---

# Part 4: Training

Two optimisers, one after the other. This is the standard PINN recipe.

**Adam** first. It is robust to a bad starting point and makes fast progress from a random
initialisation, but it stalls at a moderate loss because the landscape near the solution is badly
conditioned.

**L-BFGS** second. It is a quasi-Newton method that builds a curvature model from the gradient
history, and with a strong Wolfe line search it takes very long, well-scaled steps once it is near
the bottom. On this problem it is the optimiser that actually delivers the accuracy, and it is
cheap, because it needs few iterations.

Prediction snapshots are taken during Adam so the convergence can be animated afterwards.

In [ ]:
# --- Train the PINN ----------------------------------------------------------
LAMBDA_BC  = 10.0
N_COL      = 100
ADAM_ITERS = 2000
SNAP_EVERY = 40

model  = BeamNet(hidden=32, n_layers=3, seed=1)
x_col  = torch.linspace(0, L, N_COL).reshape(-1, 1).requires_grad_(True)
x_bc   = torch.tensor([[0.0], [L]]).requires_grad_(True)

snaps, res_snaps = [], []
t0 = time.time()
hist = train_adam(model, x_col, x_bc, LAMBDA_BC, ADAM_ITERS,
                  record=(SNAP_EVERY, snaps), record_res=res_snaps)
t_adam = time.time() - t0
rel_adam, _ = rel_l2(model)
print(f"Adam   {ADAM_ITERS} iterations in {t_adam:5.1f} s   "
      f"loss = {hist['total'][-1]:.3e}   relative L2 = {rel_adam:.3e}")

# --- L-BFGS refinement --------------------------------------------------------
opt2 = torch.optim.LBFGS(model.parameters(), lr=1.0, max_iter=300, history_size=50,
                         tolerance_grad=1e-14, tolerance_change=1e-14,
                         line_search_fn="strong_wolfe")
lbfgs_hist = []
def closure():
    opt2.zero_grad()
    lp = physics_loss(model, x_col, q_uniform)
    lb = bc_loss(model, x_bc)
    lt = lp + LAMBDA_BC * lb
    lt.backward()
    lbfgs_hist.append((lt.item(), lp.item(), lb.item()))
    if len(lbfgs_hist) % 40 == 0:
        with torch.no_grad():
            snaps.append((ADAM_ITERS + len(lbfgs_hist), model(w_plot_x).numpy().ravel()))
        dd = derivatives(model, x_col, order=4)
        res_snaps.append((EI * dd[4] - q_uniform(x_col)).detach().numpy().ravel())
    return lt

t0 = time.time()
opt2.step(closure)
t_lbfgs = time.time() - t0
rel_final, w_pinn = rel_l2(model)
with torch.no_grad():
    snaps.append((ADAM_ITERS + len(lbfgs_hist), model(w_plot_x).numpy().ravel()))
_dd = derivatives(model, x_col, order=4)
res_snaps.append((EI * _dd[4] - q_uniform(x_col)).detach().numpy().ravel())

print(f"L-BFGS {len(lbfgs_hist):4d} function evaluations in {t_lbfgs:5.1f} s   "
      f"loss = {lbfgs_hist[-1][0]:.3e}   relative L2 = {rel_final:.3e}")
print()
print(f"total training time : {t_adam + t_lbfgs:.1f} s")
print(f"accuracy improvement from L-BFGS : {rel_adam/rel_final:.0f}x")
print(f"snapshots stored for the animations : {len(snaps)} deflection, "
      f"{len(res_snaps)} residual fields")


### Watching it converge

Snapshots of $w_\theta(x)$ were stored every few dozen iterations during the run above. The
animation plays them back alongside the loss history. Use the play button; building it takes a few
seconds.

In [ ]:
# --- Animation: the deflection converging ------------------------------------
N_FRAMES = 60
sel = np.unique(np.linspace(0, len(snaps) - 1, N_FRAMES).astype(int))

adam_total = np.array(hist["total"])
lb_total   = np.array([r[0] for r in lbfgs_hist])
all_total  = np.concatenate([adam_total, lb_total])

# fixed limits on the deflection panel, set by the converged solution
W_LO, W_HI = -0.004, 0.020

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4))

ax1.plot(x_plot, w_plot_e, color=C_ALT, lw=3, alpha=0.55, label="analytical")
(line,) = ax1.plot([], [], color=C_BAD, lw=2, ls="--", label="PINN")
ax1.set_xlim(0, L); ax1.set_ylim(W_HI, W_LO)
ax1.set_xlabel("$x$  (m)"); ax1.set_ylabel("$w$  (m)")
ax1.legend(fontsize=8, loc="lower left")
title = ax1.set_title("", fontsize=10)
# The untrained network is about fifteen times the size of the solution, so the first
# frame or two would otherwise shoot off the panel and leave a stub the reader cannot
# read. The axes stay fixed on the converged solution and a note says where the curve went.
offnote = ax1.text(0.5, 0.97, "", transform=ax1.transAxes, ha="center", va="top",
                   fontsize=8, color=C_BAD,
                   bbox=dict(fc="w", ec=C_BAD, alpha=0.92, boxstyle="round,pad=0.28"))

ax2.semilogy(np.arange(1, len(all_total) + 1), all_total, color="#bbbbbb", lw=1)
(lcur,) = ax2.plot([], [], color=C_FIT, lw=1.8)
(lpt,)  = ax2.plot([], [], "o", color=C_FIT, ms=7)
ax2.axvline(ADAM_ITERS, color=C_DATA, ls="-.", lw=1.2)
ax2.text(ADAM_ITERS, all_total.max(), " Adam to L-BFGS", fontsize=8,
         color=C_DATA, va="top")
ax2.set_xlabel("iteration"); ax2.set_ylabel("total loss")
ax2.set_title("loss history", fontsize=10)

def update(f):
    it, wv = snaps[sel[f]]
    line.set_data(x_plot, wv)
    k = max(1, min(it, len(all_total)))
    lcur.set_data(np.arange(1, k + 1), all_total[:k])
    lpt.set_data([k], [all_total[k - 1]])
    err = np.linalg.norm(wv - w_plot_e) / np.linalg.norm(w_plot_e)
    title.set_text(f"iteration {it}    relative $L_2$ error = {err:.2e}")
    off = (wv.max() > W_HI) or (wv.min() < W_LO)
    msg = ""
    if off:
        msg = ("the curve leaves this panel: it spans\n"
               "w = {:.3f} to {:.3f} m, against a solution\n"
               "whose largest value is {:.4f} m".format(wv.min(), wv.max(), w_plot_e.max()))
    offnote.set_text(msg)
    return line, lcur, lpt, title, offnote

anim = animation.FuncAnimation(fig, update, frames=len(sel), interval=140, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())


**What the animation shows.** Left: the PINN deflection redrawn from stored snapshots against
the fixed analytical curve, with the iteration count and the current relative $L_2$ error in the
title. Right: the same run as a loss history, with the grey curve the full trajectory, the orange
curve the part reached so far, and the dash-dotted line marking the handover from Adam to L-BFGS.

Watch the order in which things get fixed. The network starts nearly flat, learns to bend the right
way, then pulls the ends down onto the supports, then spends the rest of the run refining a curve
that already looks right. Looking right by eye is roughly $10^{-2}$ relative error, which is the
number in the title when the curve stops visibly moving. Everything after that point is invisible in
the left panel and is orders of magnitude in the right one.

### The residual, in space

The loss history above is one number per iteration. It says nothing about **where** along the beam
the equation is not yet satisfied. The same snapshots carry that, because the residual was stored at
all 100 collocation points every time a snapshot was taken.

Three things to watch:

- the squares on the beam, one per collocation point, coloured by $|r(x_i)|$ on a logarithmic scale,
- the deflection converging on the analytical curve, exactly as before,
- the residual profile on the right, grey for the first snapshot and orange for the current one.

This is the picture that separates a PINN from a finite element solve. No linear system is assembled
and no element is integrated. There is a residual evaluated at a finite set of points, and an
optimiser pushing it down.

In [ ]:
# --- Animation: the residual field along the beam ----------------------------
RES = np.abs(np.array(res_snaps))                 # (n_snapshots, N_COL)
RES = np.maximum(RES, 1e-12)                      # keep the log colour scale finite
x_col_np = x_col.detach().numpy().ravel()
sel_r = np.unique(np.linspace(0, len(res_snaps) - 1, 60).astype(int))
R_LO, R_HI = RES.min(), RES.max()
print(f"residual magnitude over the whole run: {R_LO:.2e} to {R_HI:.2e} N/m")
print(f"frames: {len(sel_r)}")

norm = LogNorm(vmin=R_LO, vmax=R_HI)
cmap = plt.get_cmap("RdYlBu_r")

fig, (axB, axW, axR) = plt.subplots(
    1, 3, figsize=(14.5, 4.0), gridspec_kw={"width_ratios": [1.15, 1, 1], "wspace": 0.30})

# left: the beam itself, with the collocation points drawn on it
axB.set_xlim(-0.06, 1.06); axB.set_ylim(-0.42, 0.95); axB.grid(False)
axB.set_yticks([])
# the beam panel carries no numeric axis: its x tick labels would otherwise sit
# directly above the colourbar and read as a second, wrong scale for it.
axB.set_xticks([])
axB.spines["left"].set_visible(False); axB.spines["bottom"].set_visible(False)
for xa in np.linspace(0, L, 11):
    axB.annotate("", xy=(xa, 0.10), xytext=(xa, 0.50),
                 arrowprops=dict(arrowstyle="-|>", color=C_DATA, lw=1.0, mutation_scale=9))
axB.plot([0, L], [0.50, 0.50], color=C_DATA, lw=1.2)
axB.text(L/2, 0.70, "$q(x) = q_0$  (N/m)", ha="center", fontsize=10, color=C_DATA)
axB.plot([0, L], [0, 0], color="0.55", lw=6, zorder=1, solid_capstyle="butt")
sc = axB.scatter(x_col_np, np.zeros_like(x_col_np), c=RES[0], cmap=cmap, norm=norm,
                 s=46, marker="s", edgecolors="none", zorder=4)
axB.plot([0, L], [-0.10, -0.10], marker="^", ls="none", ms=12, color="0.35", zorder=2)
axB.text(0, -0.26, "$x=0$", ha="center", fontsize=9)
axB.text(L, -0.26, "$x=L$", ha="center", fontsize=9)
cb = fig.colorbar(sc, ax=axB, orientation="horizontal", pad=0.06, fraction=0.07, aspect=32)
cb.set_label(r"$|r| = |EI\,w'''' - q|$   (N/m)", fontsize=9)
cb.ax.grid(False)
tB = axB.set_title("", fontsize=10)

# middle: the deflection, on the same fixed axes as the previous animation
axW.plot(x_plot, w_plot_e, color=C_ALT, lw=3, alpha=0.55, label="analytical")
(lwc,) = axW.plot([], [], color=C_BAD, lw=2, ls="--", label="PINN")
axW.set_xlim(0, L); axW.set_ylim(W_HI, W_LO)
axW.set_xlabel("$x$  (m)"); axW.set_ylabel("$w$  (m)")
axW.legend(fontsize=8, loc="lower left")
tW = axW.set_title("", fontsize=10)
offn = axW.text(0.5, 0.97, "", transform=axW.transAxes, ha="center", va="top", fontsize=8,
                color=C_BAD, bbox=dict(fc="w", ec=C_BAD, alpha=0.92, boxstyle="round,pad=0.28"))

# right: the residual profile, first snapshot left in grey for reference
axR.semilogy(x_col_np, RES[0], color="0.78", lw=1.1, label="at the first snapshot")
(lrc,) = axR.semilogy([], [], color=C_FIT, lw=1.7, label="now")
axR.set_xlim(0, L); axR.set_ylim(R_LO * 0.4, R_HI * 4)
axR.set_xlabel("$x$  (m)"); axR.set_ylabel("$|r(x)|$  (N/m)")
axR.legend(fontsize=8, loc="lower center")
tR = axR.set_title("", fontsize=10)

def update_res(f):
    i = sel_r[f]
    r = RES[i]; it, wv = snaps[i]
    sc.set_array(r); lwc.set_data(x_plot, wv); lrc.set_data(x_col_np, r)
    tB.set_text("iteration %d   (%s)" % (it, "L-BFGS" if it > ADAM_ITERS else "Adam"))
    tW.set_text("relative $L_2$ error = %.2e"
                % (np.linalg.norm(wv - w_plot_e) / np.linalg.norm(w_plot_e)))
    tR.set_text("mean $|r|$ = %.1e     max $|r|$ = %.1e" % (r.mean(), r.max()))
    off = (wv.max() > W_HI) or (wv.min() < W_LO)
    offn.set_text("" if not off else
                  "the curve leaves this panel:\nit spans $w$ = %.3f to %.3f m"
                  % (wv.min(), wv.max()))
    return sc, lwc, lrc

anim_res = animation.FuncAnimation(fig, update_res, frames=len(sel_r), interval=140, blit=False)
plt.close(fig)
HTML(anim_res.to_jshtml())


**What the animation showed.** Left: the beam under its uniform load, with the 100 collocation
points drawn on it as squares coloured by the local residual magnitude, red for large and blue for
small. Middle: the deflection. Right: the residual along the beam, grey at the first snapshot and
orange now.

The residual starts at about $q_0$ everywhere, which is what a nearly flat network gives, because
then $EI\,w'''' \approx 0$ and the whole load is left over. It does not come down uniformly. The
interior falls first, the ends lag, and the orange profile develops deep notches where the network
happens to satisfy the equation exactly, with peaks between them. In the last frames the largest
residual is still sitting near the right-hand support, and the beam strip shows it as one darker
square in an otherwise pale row. The mean and maximum for each frame are printed above the right
panel.

That unevenness is not a defect of this run. It is what minimising a residual at points looks like.
A finite element solve of the same beam satisfies the equation in a weighted average sense over every
element by construction, and what is left is discretisation error with a known convergence rate.
Here the equation is satisfied at the points you chose, to whatever accuracy a non-convex optimiser
happened to reach, with no rate and no bound. Part 5 turns that observation into a failure.

### What the trained network gives you

The network was only ever told about $w''''$. Differentiating it once, twice and three times
recovers the slope, the bending moment and the shear force, and none of those were in the loss.
That is a genuine advantage of the representation: the solution is a differentiable function, not a
table of nodal values, so derived quantities come out without post-processing.

In [ ]:
# --- Deflection, moment, shear, and the residual -----------------------------
xe = torch.from_numpy(x_plot).float().reshape(-1, 1).requires_grad_(True)
d  = derivatives(model, xe, order=4)
w_p  =  d[0].detach().numpy().ravel()
M_p  = (-EI * d[2]).detach().numpy().ravel()
V_p  = (-EI * d[3]).detach().numpy().ravel()
res  = (EI * d[4] - q_uniform(xe)).detach().numpy().ravel()

def report(pred, exact, name):
    r = np.linalg.norm(pred - exact) / np.linalg.norm(exact)
    print(f"  {name:22s} max |error| = {np.max(np.abs(pred-exact)):.3e}   relative L2 = {r:.3e}")
    return r

print("PINN against the closed form:")
r_w = report(w_p, w_exact(x_plot), "deflection w")
r_m = report(M_p, M_exact(x_plot), "moment  -EI w''")
r_v = report(V_p, V_exact(x_plot), "shear   -EI w'''")
print()
print(f"  mid-span deflection, PINN  : {np.interp(L/2, x_plot, w_p):.8f} m")
print(f"  mid-span deflection, exact : {w_max_closed:.8f} m")

fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
for ax, p, e, ttl, yl in [
        (axes[0], w_p, w_exact(x_plot), "deflection $w$", "$w$ (m)"),
        (axes[1], M_p, M_exact(x_plot), "moment $M=-EIw''$", "$M$ (N m)"),
        (axes[2], V_p, V_exact(x_plot), "shear $V=-EIw'''$", "$V$ (N)")]:
    ax.plot(x_plot, e, color=C_ALT, lw=3, alpha=0.55, label="analytical")
    ax.plot(x_plot, p, color=C_BAD, lw=1.5, ls="--", label="PINN")
    ax.set_xlabel("$x$ (m)"); ax.set_ylabel(yl); ax.set_title(ttl, fontsize=10)
axes[0].invert_yaxis(); axes[0].legend(fontsize=8)

axes[3].plot(x_plot, res, color=C_FIT, lw=1.4)
axes[3].axhline(0, color="k", lw=0.8)
axes[3].set_xlabel("$x$ (m)"); axes[3].set_ylabel("$EIw'''' - q$")
axes[3].set_title(f"PDE residual\nmax |r| = {np.max(np.abs(res)):.1e}", fontsize=10)
plt.tight_layout(); plt.show()


**What the four panels show.** The first three are the deflection, the bending moment and the
shear force, PINN against analytical, with the relative $L_2$ errors printed above. Only the
deflection was ever in the loss. The moment and the shear come out of differentiating the same
network twice and three times, with no post-processing and no recovery procedure. The fourth panel
is the residual $r(x)$ along the beam, the quantity the training actually minimised.

The residual is not uniform along the beam. It is largest near the ends, where the boundary
terms pull against the interior equation. That competition is the same weighting problem from Part 3,
now visible in space rather than as a single number.

### How the network assembles the solution

The last layer is linear. Write $h_j(x)$ for the $j$-th activation of the last hidden layer and
$W_j$ for the weight that reads it. Then

$$\boxed{\;w_\theta(x) \;=\; \sum_{j=1}^{H} W_j\,h_j(x) \;+\; b\;}$$

with $H = 32$ here. Once training has finished every $h_j$ is a fixed smooth function of $x$, so the
trained network is a linear combination of 32 learned functions of position. That is in the spirit
of a basis expansion.

It is not the same object as a finite element basis or a spectral basis, and this audience will know
why. The functions were not chosen in advance, they have no interpolation or partition-of-unity
property, they are not orthogonal, they have global support, and no approximation theory attaches to
them. What follows measures the difference rather than asserting it.

The animation adds the 32 weighted terms one at a time, in index order. Watch the middle panel: the
running sum does not creep up on the answer.

In [ ]:
# --- Take the trained network apart --------------------------------------
X_b = torch.from_numpy(x_plot).float().reshape(-1, 1)
with torch.no_grad():
    h_b = X_b
    for layer in list(model.net)[:-1]:          # everything except the final Linear
        h_b = layer(h_b)
    H_act = h_b.numpy()                          # (300, 32) last hidden layer activations
    last_layer = list(model.net)[-1]
    W_last = last_layer.weight.detach().numpy().ravel()
    b_last = last_layer.bias.detach().numpy().item()
    w_net = model(X_b).numpy().ravel()

CONTRIB = H_act * W_last[None, :]                # (300, 32) weighted contributions, in m
print(f"identity check, |sum_j W_j h_j + b - w_theta|_max = "
      f"{np.abs(CONTRIB.sum(1) + b_last - w_net).max():.2e} m")

peak_w   = np.abs(w_net).max()
peak_c   = np.abs(CONTRIB).max()
print(f"peak deflection of the trained network : {peak_w:.6f} m")
print(f"largest single weighted contribution   : {peak_c:.6f} m   "
      f"({peak_c/peak_w:.1f}x the peak deflection)")
print(f"the output bias b                      : {b_last:.6f} m   "
      f"({abs(b_last)/peak_w:.1f}x the peak deflection)")
print()

# running partial sums, in index order
PSUM = np.hstack([np.full((len(x_plot), 1), b_last),
                  b_last + np.cumsum(CONTRIB, axis=1)])         # (300, 33)
err_sum = np.array([np.linalg.norm(PSUM[:, k] - w_net) / np.linalg.norm(w_net)
                    for k in range(33)])

# best k-term least squares fit to the exact deflection, from the same 32 functions
ONE = np.ones((len(x_plot), 1))
err_ls = []
for k in range(33):
    Ak = np.hstack([ONE, H_act[:, :k]]) if k > 0 else ONE
    ck, *_ = np.linalg.lstsq(Ak, w_plot_e, rcond=None)
    err_ls.append(np.linalg.norm(Ak @ ck - w_plot_e) / np.linalg.norm(w_plot_e))
err_ls = np.array(err_ls)
for k in [1, 2, 4, 6, 8, 16, 32]:
    print(f"  least squares fit to the exact solution using {k:2d} of the 32 functions: "
          f"relative L2 = {err_ls[k]:.2e}")


In [ ]:
# --- Animation: the solution assembled term by term --------------------------
n_H = CONTRIB.shape[1]
C_LO = min(CONTRIB.min(), b_last) * 1.15
C_HI = max(CONTRIB.max(), b_last) * 1.15
S_LO = min(PSUM.min(), w_net.min()); S_HI = max(PSUM.max(), w_net.max())
_pad = 0.08 * (S_HI - S_LO); S_LO -= _pad; S_HI += _pad
kk = np.arange(n_H + 1)

fig, (b1, b2, b3) = plt.subplots(1, 3, figsize=(14.5, 4.0), gridspec_kw={"wspace": 0.30})

faint = [b1.plot([], [], color="0.78", lw=0.9)[0] for _ in range(n_H)]
(curln,) = b1.plot([], [], color=C_FIT, lw=2.4)
(biasln,) = b1.plot([0, L], [b_last, b_last], color=C_DATA, lw=1.6, ls=":")
b1.axhline(0, color="k", lw=0.8)
b1.set_xlim(0, L); b1.set_ylim(C_LO, C_HI)
b1.set_xlabel("$x$  (m)"); b1.set_ylabel("$W_j\\,h_j(x)$  (m)")
tb1 = b1.set_title("", fontsize=10)

b2.plot(x_plot, w_net, color=C_ALT, lw=3, alpha=0.6, label="$w_\\theta(x)$, all 32 terms")
(lsum,) = b2.plot([], [], color=C_BAD, lw=2, ls="--", label="running sum")
b2.set_xlim(0, L); b2.set_ylim(S_HI, S_LO)
b2.set_xlabel("$x$  (m)"); b2.set_ylabel("$w$  (m)")
b2.legend(fontsize=8, loc="upper left")
b2.text(0.98, 0.04, "the solution's peak is %.4f m,\nwhich is nearly flat at this scale" % peak_w,
        transform=b2.transAxes, ha="right", va="bottom", fontsize=8, color=C_ALT)
tb2 = b2.set_title("", fontsize=10)

b3.semilogy(kk, np.maximum(err_sum, 1e-7), "o-", color=C_BAD, lw=1.4, ms=3.5,
            label="the network's own running sum")
b3.semilogy(kk, np.maximum(err_ls, 1e-7), "s-", color=C_DATA, lw=1.4, ms=3.5,
            label="best $k$-term least squares fit")
(mkr,) = b3.plot([], [], "o", color="k", ms=9, mfc="none", mew=1.6)
b3.set_xlim(-0.5, n_H + 0.5); b3.set_ylim(5e-8, 2e3)
b3.set_xlabel("terms included, $k$"); b3.set_ylabel("relative $L_2$ difference")
b3.legend(fontsize=7, loc="upper center", ncol=2, framealpha=0.92)
b3.set_title("two ways to use the same 32 functions", fontsize=10)

def update_basis(k):
    for j in range(n_H):
        if j < k:
            faint[j].set_data(x_plot, CONTRIB[:, j])
        else:
            faint[j].set_data([], [])
    if k >= 1:
        curln.set_data(x_plot, CONTRIB[:, k - 1])
        tb1.set_text("weighted activation $j$ = %d of %d" % (k, n_H))
    else:
        curln.set_data([], [])
        tb1.set_text("the output bias $b$ alone (dotted)")
    lsum.set_data(x_plot, PSUM[:, k])
    mkr.set_data([k], [max(err_sum[k], 6e-8)])
    tb2.set_text("%d terms,  difference from $w_\\theta$ = %.2e" % (k, err_sum[k]))
    return faint + [curln, lsum, mkr]

anim_basis = animation.FuncAnimation(fig, update_basis, frames=n_H + 1, interval=180, blit=False)
plt.close(fig)
HTML(anim_basis.to_jshtml())


**What the animation showed.** Left: the weighted activations $W_j h_j(x)$, the one being
added drawn in orange and the ones already added in grey, with the output bias dotted. Middle: the
running sum against the finished network output, on axes wide enough for the whole excursion.
Right: two error curves against the number of terms, red for the network's own partial sums and blue
for the best least squares fit that the same functions can produce.

The middle panel is the point. The partial sums swing well outside the deflection, and the running
sum stays several times the size of the solution until the very last term, when the whole thing
collapses onto the answer. The printed ratios above say why: the largest single contribution and the
output bias are both an order of magnitude larger than the peak deflection. The network reaches a
deflection of order $10^{-2}$ m by cancelling terms of order $10^{-1}$ m against each other.

The blue curve in the right panel says something different and equally worth knowing. Fit the same
32 functions to the exact solution by least squares and the error falls quickly: the printed table
gives the value at 4, 6 and 8 functions. So the learned basis is expressive, and it is the
optimiser's particular choice of coefficients that is near-cancelling. A finite element basis
behaves neither way. Each of its terms is a nodal value of the answer, so the partial sums stay the
size of the answer, and the coefficients are determined by a linear solve rather than by gradient
descent.

In [ ]:
# --- The learned basis itself ------------------------------------------------
fig, (g1, g2) = plt.subplots(1, 2, figsize=(11.5, 3.9))

dH = np.diff(H_act, axis=0)
n_mono = int(np.sum(np.all(dH >= 0, axis=0) | np.all(dH <= 0, axis=0)))
curv = np.abs(np.diff(H_act, 2, axis=0)).max(0) / (np.ptp(H_act, axis=0) + 1e-12)
print(f"of the {n_H} activations, {n_mono} are monotone in x over [0, L]")
print(f"largest normalised second difference across the 32: {curv.max():.2e}")

cols = plt.get_cmap("viridis")(np.linspace(0, 0.92, n_H))
for j in range(n_H):
    g1.plot(x_plot, H_act[:, j], lw=1.0, alpha=0.9, color=cols[j])
g1.set_xlabel("$x$  (m)"); g1.set_ylabel("$h_j(x)$  (dimensionless)")
g1.set_title(f"the {n_H} last-layer activations of the trained PINN", fontsize=10)

k_show = 4
A4 = np.hstack([ONE, H_act[:, :k_show]])
c4, *_ = np.linalg.lstsq(A4, w_plot_e, rcond=None)
g2.plot(x_plot, w_plot_e, color=C_ALT, lw=3, alpha=0.6, label="analytical")
g2.plot(x_plot, A4 @ c4, color=C_BAD, lw=1.6, ls="--",
        label=f"{k_show} of the 32 functions,\nleast squares (rel $L_2$ = {err_ls[k_show]:.1e})")
g2.invert_yaxis()
g2.set_xlabel("$x$  (m)"); g2.set_ylabel("$w$  (m)")
g2.set_title("what a handful of them can represent", fontsize=10)
g2.legend(fontsize=7.5, loc="upper center")
plt.tight_layout(); plt.show()


**What the two panels show.** Left: all 32 activations of the last hidden layer, plotted as
functions of position. This is the learned basis. Each one is smooth, the count of monotone ones is
printed above, and over this domain they stay close to straight, because the argument of the final
$\tanh$ never leaves its near-linear range. They look nothing like hat functions or sine modes, and
nothing asked them to. Right: the least squares combination of the
first four of them against the analytical deflection, with its error in the legend.

Four ramps and a constant already reproduce a quartic to the accuracy in that legend. That is a fair
description of what the network has available to it. What it does with that availability is the
middle panel of the animation.

---

# Part 5: How many collocation points?

The residual is enforced only where you ask. With too few points, the network can satisfy the
equation at every point it is shown and do whatever it likes in between. This is overfitting, with
collocation points in the role that training data played in Notebooks 1 to 4.

The sweep below trains six PINNs with 2 to 20 collocation points each, on an identical budget of
1200 Adam iterations plus L-BFGS. Watch the two numbers printed for each run: the **training loss**
and the **true error**.

In [ ]:
# --- Collocation sweep --------------------------------------------------------
N_SWEEP    = [2, 3, 4, 5, 8, 20]
COL_ITERS  = 1200
colloc = {}

t_cs = time.time()
for nc in N_SWEEP:
    m  = BeamNet(seed=2)
    xc = torch.linspace(0, L, nc).reshape(-1, 1).requires_grad_(True)
    xb = torch.tensor([[0.0], [L]]).requires_grad_(True)
    train_adam(m, xc, xb, LAMBDA_BC, COL_ITERS)
    o  = torch.optim.LBFGS(m.parameters(), lr=1.0, max_iter=300, history_size=50,
                           line_search_fn="strong_wolfe")
    def cl(m=m, xc=xc, xb=xb):
        o.zero_grad()
        lt = physics_loss(m, xc, q_uniform) + LAMBDA_BC * bc_loss(m, xb)
        lt.backward(); return lt
    o.step(cl)

    with torch.no_grad():
        wv = m(w_plot_x).numpy().ravel()
    train_loss = (physics_loss(m, xc, q_uniform) + LAMBDA_BC * bc_loss(m, xb)).item()
    # residual on a dense grid the network never trained on
    xd = torch.linspace(0, L, 300).reshape(-1, 1).requires_grad_(True)
    dd = derivatives(m, xd, order=4)
    res_d = (EI * dd[4] - q_uniform(xd)).detach().numpy().ravel()
    r = np.linalg.norm(wv - w_plot_e) / np.linalg.norm(w_plot_e)
    colloc[nc] = {"w": wv, "rel": r, "loss": train_loss,
                  "res": res_d, "xc": xc.detach().numpy().ravel()}
    print(f"  N_c = {nc:3d}   training loss = {train_loss:.2e}   "
          f"true relative L2 error = {r:.3e}", flush=True)
print(f"\n{len(N_SWEEP)} runs in {time.time()-t_cs:.1f} s")


Read the two printed columns against each other. The runs with the fewest points have the
smallest training losses and the largest true errors. With two collocation points the training loss
is among the lowest in the sweep and the solution is wrong by more than 100 percent. The optimiser
did exactly what it was asked. It was asked the wrong question.

The third panel below is the diagnostic that catches this: evaluate the residual on a dense grid the
network never saw. Where the dense residual is large between the training points, the collocation
set is too sparse. It is the PINN equivalent of a held-out test set, and it costs one forward pass.

In [ ]:
# --- Interactive: collocation points -----------------------------------------
def show_colloc(n_points=20):
    d = colloc[n_points]
    fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(14, 3.8))

    a1.plot(x_plot, w_plot_e, color=C_ALT, lw=3, alpha=0.6, label="analytical")
    a1.plot(x_plot, d["w"],   color=C_BAD, lw=2, ls="--", label="PINN")
    a1.scatter(d["xc"], np.zeros_like(d["xc"]), s=45, color=C_FIT, zorder=5,
               label=f"{n_points} collocation points")
    a1.invert_yaxis(); a1.set_ylim(0.06, -0.06)
    a1.set_xlabel("$x$ (m)"); a1.set_ylabel("$w$ (m)")
    a1.set_title(f"relative $L_2$ error = {d['rel']:.2e}", fontsize=10)
    a1.legend(fontsize=7, loc="lower center")

    a2.semilogy(N_SWEEP, [colloc[k]["loss"] for k in N_SWEEP], "o-",
                color=C_DATA, lw=2, label="training loss")
    a2.semilogy(N_SWEEP, [colloc[k]["rel"] for k in N_SWEEP], "s-",
                color=C_BAD, lw=2, label="true error")
    a2.axvline(n_points, color="k", ls="--", lw=1)
    a2.set_xlabel("number of collocation points"); a2.set_ylabel("value")
    a2.set_title("loss says one thing, error says another", fontsize=10)
    a2.legend(fontsize=8)

    a3.semilogy(x_plot, np.abs(d["res"]) + 1e-16, color=C_FIT, lw=1.3)
    a3.scatter(d["xc"], np.full_like(d["xc"], 1e-14), s=45, color=C_BAD,
               marker="^", zorder=5, clip_on=False)
    a3.set_ylim(1e-14, 1e3)
    a3.set_xlabel("$x$ (m)"); a3.set_ylabel("$|EIw'''' - q|$")
    a3.set_title("residual on a dense grid\n(triangles: training points)", fontsize=10)
    plt.tight_layout(); plt.show()

interact(show_colloc,
         n_points=SelectionSlider(options=N_SWEEP, value=20,
                                  continuous_update=False, description="N points"));


**What the three panels show, and this is the figure to stare at.** Left: the deflection for
the selected number of collocation points, with those points drawn on the axis. Middle: two curves
against point count, the blue training loss and the red true error. At the left-hand end they move
in opposite directions. Right: the residual on a dense grid, with the training points marked as
triangles.

Set the slider to 2 and read the middle panel. The training loss is among the lowest of any run and
the true error is the highest. The network satisfied the equation exactly at both points it was
shown and did as it pleased in between. The right panel is where that becomes visible: the residual
dives towards zero at each triangle and is large everywhere else.

This is the PINN version of overfitting, and it is more dangerous than the version in Notebooks 1 to
4, because there is no held-out label to catch it. The loss you watch during training is the loss on
the points you chose. The dense residual in the right panel costs one forward pass and is the check
that does catch it.

A caveat, and it matters. This particular solution is a quartic polynomial, which is about the
easiest thing a smooth network can represent. Five well-placed points nearly pin it down. For a
problem with a boundary layer, a shock, or a high-frequency solution, the required point count rises
sharply and uniform spacing stops being sensible. Adaptive resampling, concentrating points where
the residual is large, is the usual remedy.

---

# Part 6: Where PINNs struggle

Four honest limitations, in increasing order of how much they should worry you.

**1. The loss weight is a free parameter.** Part 3 measured it: the same network and the same
budget gave errors spanning four orders of magnitude across the weights tried, with the spread
printed by the sweep. A classical solver has no such knob.

**2. Convergence is not guaranteed.** Training is non-convex optimisation. There is no residual
bound, no mesh refinement argument, no error estimate. A PINN that has converged to a small loss may
still be wrong, as the two-point case in Part 5 showed.

**3. Higher-order derivatives are expensive.** Each `autograd.grad` call deepens the graph. The
next cell measures the cost as a function of derivative order for this network.

**4. The network solves one problem.** This is the important one. $w_\theta$ is trained against one
load, one set of boundary conditions, one geometry. Change any of them and the weights are worthless.
There is no sense in which the PINN has learned beam bending; it has learned this beam under this
load. The cell after next measures what that costs.

In [ ]:
# --- The cost of each derivative order ---------------------------------------
xb_t = torch.linspace(0, L, 100).reshape(-1, 1).requires_grad_(True)
print("cost of one forward pass plus backward, by derivative order")
print("(100 points, 32x3 network, mean of 60 repeats)")
base = None
for order in range(5):
    t0 = time.time()
    for _ in range(60):
        dd = derivatives(model, xb_t, order=order)
        dd[-1].sum().backward()
    dt = (time.time() - t0) / 60 * 1000
    if order == 0: base = dt
    print(f"  order {order}:  {dt:7.2f} ms   ({dt/base:5.1f}x the plain forward pass)")


**What the timing table shows.** Order 0 is a plain forward pass plus backward, the cost any
network in Notebooks 1 to 4 would pay. Each further row adds one `autograd.grad` call, and the
multiplier in brackets is measured against that baseline. The growth is steep because every
differentiation deepens the graph that then has to be traversed backwards. For a fourth-order
equation you pay this on every optimiser step, which is why the small network above is not a
compromise: width is cheap here, derivative order is not.

### The retrain

Now change the load. Same beam, same supports, same network architecture, same hyperparameters.
Only $q(x)$ changes, from uniform to $q_0\sin(\pi x/L)$.

The closed form for the sine load on a simply supported beam is

$$w(x) = \frac{q_0 L^4}{\pi^4 EI}\,\sin\!\left(\frac{\pi x}{L}\right)$$

which is verified below before it is used. Then the trained uniform-load network is evaluated on
the new problem, and a fresh network is trained from scratch and timed.

In [ ]:
# --- A different load --------------------------------------------------------
def q_sine(x):
    return q0 * torch.sin(np.pi * x / L)

def w_exact_sine(x):
    return (q0 * L**4 / (np.pi**4 * EI)) * np.sin(np.pi * x / L)

# verify the closed form the same way as in Part 1
q_grid   = q0 * np.sin(np.pi * xg / L)
u_s = np.zeros(n); u_s[1:-1] = np.linalg.solve(A, q_grid[1:-1])
w_s = np.zeros(n); w_s[1:-1] = np.linalg.solve(A, u_s[1:-1] / EI)
w_s_cf = w_exact_sine(xg)
print(f"sine load, finite difference vs closed form:  relative L2 = "
      f"{np.linalg.norm(w_s - w_s_cf)/np.linalg.norm(w_s_cf):.3e}")

# how does the already-trained network do on the new load?
w_sine_e = w_exact_sine(x_plot)
with torch.no_grad():
    w_old = model(w_plot_x).numpy().ravel()
rel_transfer = np.linalg.norm(w_old - w_sine_e) / np.linalg.norm(w_sine_e)
print(f"the uniform-load PINN applied to the sine load: relative L2 = {rel_transfer:.3f}")


In [ ]:
# --- Retrain from scratch, and time it ---------------------------------------
model_s = BeamNet(hidden=32, n_layers=3, seed=1)
x_col_s2 = torch.linspace(0, L, N_COL).reshape(-1, 1).requires_grad_(True)
x_bc_s2  = torch.tensor([[0.0], [L]]).requires_grad_(True)

t_retrain = time.time()
train_adam(model_s, x_col_s2, x_bc_s2, LAMBDA_BC, ADAM_ITERS, q_fn=q_sine)
opt_s = torch.optim.LBFGS(model_s.parameters(), lr=1.0, max_iter=300, history_size=50,
                          line_search_fn="strong_wolfe")
def cl_s():
    opt_s.zero_grad()
    lt = physics_loss(model_s, x_col_s2, q_sine) + LAMBDA_BC * bc_loss(model_s, x_bc_s2)
    lt.backward(); return lt
opt_s.step(cl_s)
t_retrain = time.time() - t_retrain

with torch.no_grad():
    w_new = model_s(w_plot_x).numpy().ravel()
rel_new = np.linalg.norm(w_new - w_sine_e) / np.linalg.norm(w_sine_e)

print("=" * 62)
print(f"  RETRAIN FOR ONE NEW LOAD CASE : {t_retrain:.1f} s")
print(f"  accuracy after retraining     : relative L2 = {rel_new:.3e}")
print(f"  accuracy without retraining   : relative L2 = {rel_transfer:.3f}")
print("=" * 62)
print()
print(f"A parametric study over 50 load cases would cost about "
      f"{50*t_retrain/60:.0f} minutes at this rate,")
print("and every one of those runs would rediscover the same beam theory from scratch.")


In [ ]:
# --- The picture that sets up Notebook 6 -------------------------------------
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 4))

a1.plot(x_plot, w_sine_e, color=C_ALT, lw=3, alpha=0.6, label="analytical, sine load")
a1.plot(x_plot, w_old, color=C_BAD, lw=2, ls="--",
        label=f"uniform-load PINN, no retrain\n(rel $L_2$ = {rel_transfer:.2f})")
a1.plot(x_plot, w_new, color=C_DATA, lw=2, ls=":",
        label=f"retrained PINN\n(rel $L_2$ = {rel_new:.1e})")
a1.invert_yaxis()
a1.set_xlabel("$x$ (m)"); a1.set_ylabel("$w$ (m)")
a1.set_title("change the load, and the old weights are worthless", fontsize=10)
a1.legend(fontsize=7, loc="upper center", framealpha=0.95)

a2.bar(["uniform\n(Part 4)", "sine\n(retrained)"],
       [t_adam + t_lbfgs, t_retrain], color=[C_DATA, C_FIT], width=0.5)
a2.set_ylabel("training time (s)")
a2.set_title("one trained network, one load case", fontsize=10)
for i, v in enumerate([t_adam + t_lbfgs, t_retrain]):
    a2.text(i, v, f"{v:.0f} s", ha="center", va="bottom", fontsize=10)
plt.tight_layout(); plt.show()


**What the two panels show.** Left: three curves on the sine-load problem. Green is the
analytical answer. The red dashed curve is the network trained in Part 4 on the uniform load,
evaluated without retraining, and it is simply the wrong shape, with its relative error printed in
the legend. The blue dotted curve is a fresh network trained on the sine load, which lands back on
the exact solution. There is no partial credit here: the old weights are not a warm start, they are
wrong.

Right: what that cost. Two bars, two load cases, two complete training runs, with the seconds
printed on each. Nothing was reused between them. Scale that bar chart to a parametric study and the
problem with PINNs as surrogates is the whole picture.

### Where this leaves us, and what Notebook 6 does about it

A PINN is a solver, not a surrogate. It replaces the discretisation, not the simulation campaign.
Compared with a finite element solve of the same beam it is slower, less accurate and carries no
error bound, and the honest summary is that for a linear ODE on a one-dimensional domain, finite
elements win comfortably. PINNs earn their place on problems where meshing is genuinely hard, where
the equation is known but the coefficients are not, or where sparse measurements must be fused with
physics. Those are real, and they are not this beam.

The limitation that does not go away is the last one. Every new load, every new geometry, every new
boundary condition is a new optimisation run.

Notebook 6 changes the object being learned. Instead of learning a function $x \mapsto w(x)$ for one
load, a neural operator learns a mapping $q \mapsto w$ between whole functions. Train it once over a
family of loads, and a new load case is one forward pass rather than the retrain timed above.

---

# What to take away

1. A PINN is trained against the governing equation, not against data. The residual and the boundary
   terms are the entire loss, and the beam above was solved without a single labelled example. The
   network plays the role the discretisation plays in a classical solver, and Part 4 took the trained
   one apart to show what that basis actually is.
2. Automatic differentiation gives exact derivatives of the network with respect to its inputs, to
   any order. Part 2 checked this to machine precision at fourth order, and the cost measured in
   Part 6 grows steeply with order.
3. The relative weight between the physics loss and the boundary loss is a hyperparameter you must
   choose, and the wrong choice produces a confidently wrong answer with a small loss.
4. A small training loss is not evidence of a correct solution. Two collocation points drove the
   training loss below $10^{-9}$ while the deflection was wrong by more than 100 percent. Check the
   residual on points the network never trained on.
5. Adam gets close, L-BFGS gets it right. The two-phase recipe is standard for a reason, and the
   refinement phase is cheap.
6. The trained network solves exactly one problem. Changing the load meant retraining from scratch,
   at the cost printed in Part 6. That is the limitation Notebook 6 is built to remove.

---

# Exercises

### 1. Change the rigidity
Set `EI = 2.0` and rerun Part 4. Before you do, write down what should happen to the mid-span
deflection. Does the PINN reproduce the factor you predicted, and does the training time change?

### 2. Read the weighting off the units
Part 3 argued that $L_{phys}$ and $L_{bc}$ differ in scale by roughly
$(EI/L^4)^2$. Non-dimensionalise the boundary loss by dividing the deflection terms by
$q_0L^4/EI$ and the curvature terms by $q_0L^2/EI$, then repeat the $\lambda$ sweep. Does the best
weight move towards 1?

### 3. Clamp both ends
Replace the simply supported conditions with $w(0)=w(L)=0$ and $w'(0)=w'(L)=0$. Derive the closed
form, verify it against the finite difference solve in Part 1, and train a PINN for it. Which of the
two problems is harder for the optimiser, and why might that be?

### 4. Move the points
Part 5 used uniformly spaced collocation points. Retrain with 8 points drawn at random from a
uniform distribution, using several different seeds. How much does the final error vary between
seeds, and what does that tell you about reporting a single PINN result?

### 5. The simple method wins
Solve the same beam with the finite difference scheme from Part 1 at 401 nodes, and time it. Compare
the wall-clock time and the relative $L_2$ error against the PINN in Part 4. Which method would you
use for this problem, and what would have to change about the problem before your answer changed?

### 6. Make the retrain cheaper
Instead of reinitialising, start the sine-load training from the converged uniform-load weights and
time it. How much does warm starting save, and does the saving survive if the new load is very
different from the old one? What does your answer suggest about the limits of transfer learning as a
fix for limitation 4?
